# Perform NicheNet Analysis Starting from an AnnData Object: Step-by-Step Analysis

In this notebook, you can learn how to perform a basic NicheNet analysis on an AnnData object containing single-cell expression data. The steps of this notebook can also be adapted for other single-cell or bulk frameworks.

**Assuming you have captured the changes in gene expression resulting from your cell-cell communication (CCC) process of interest,** a NicheNet analysis can help you to generate hypotheses about the CCC process. Specifically, NicheNet can predict 1) which ligands from the microenvironment or cell population(s) ("sender/niche") are most likely to affect target gene expression in an interacting cell population ("receiver/target") and 2) which specific target genes are affected by which of these predicted ligands.

As example expression data of interacting cells, we will use mouse NICHE-seq data to explore intercellular communication in the T cell area in the inguinal lymph node before and 72 hours after lymphocytic choriomeningitis virus (LCMV) infection (Medaglia et al., 2017). We will focus on CD8 T cells as the receiver population, and as this dataset contains two conditions (before and after LCMV infection), the differentially expressed genes between these two conditions in CD8 T cells will be used as our gene set of interest. We will then prioritize which ligands from the microenvironment (sender-agnostic approach) and from specific immune cell populations like monocytes, dendritic cells, NK cells, B cells, and CD4 T cells (sender-focused approach) can regulate and induce these observed gene expression changes.

## Prepare NicheNet Analysis

### Load packages and set up data directory

In [ ]:
import os
os.environ.setdefault("NICHENETR_DATA_DIR", "path/to/nichenetr_data")

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import nichenetr as nn

### Read in the expression data of interacting cells

We load a processed AnnData object (the Python equivalent of the Seurat object from the original vignette). Note that genes should be named by their official mouse/human gene symbol.

In [ ]:
adata = nn.load_seurat_obj()
adata.obs.head()

In [ ]:
# Convert gene aliases to official symbols
adata = nn.alias_to_symbol_anndata(adata, "mouse")

In [ ]:
# Visualize cell populations
print(adata.obs["celltype"].value_counts())
sc.pl.tsne(adata, color="celltype")

In [ ]:
# Visualize conditions
print(adata.obs["aggregate"].value_counts())
sc.pl.tsne(adata, color="aggregate")

### Read in NicheNet's networks

The ligand-target prior model, ligand-receptor network, and weighted integrated networks are needed. The ligand-target prior model is a matrix describing the potential that a ligand may regulate a target gene. The ligand-receptor network contains information on potential ligand-receptor bindings. The weighted ligand-receptor network contains weights representing the potential that a ligand will bind to a receptor.

In [ ]:
organism = "mouse"

lr_network = nn.load_lr_network(organism)
ligand_target_matrix = nn.load_ligand_target_matrix(organism)
weighted_networks = nn.load_weighted_networks(organism)

# Keep distinct ligand-receptor pairs
lr_network = lr_network[["from", "to"]].drop_duplicates()
print(lr_network.head())

# Inspect the ligand-target matrix
print(f"Ligand-target matrix shape: {ligand_target_matrix.data.shape}")
print(f"Number of target genes (rows): {len(ligand_target_matrix.rownames)}")
print(f"Number of ligands (cols): {len(ligand_target_matrix.colnames)}")

# Inspect weighted networks
print("\nlr_sig network:")
print(weighted_networks["lr_sig"].head())
print("\ngr network:")
print(weighted_networks["gr"].head())

## Perform the NicheNet Analysis

We now recommend users to run both the "sender-agnostic" approach and "sender-focused" approach. These approaches only affect the list of potential ligands that are considered for prioritization.

### 1. Define a set of potential ligands

We first define a "receiver/target" cell population and determine which genes are expressed. Here, we consider a gene to be expressed if it is expressed in at least 5% of cells. The receiver cell population is CD8 T cells.

In [ ]:
receiver = "CD8 T"
expressed_genes_receiver = nn.get_expressed_genes(
    adata, celltype_col="celltype", celltype=receiver, pct=0.05
)
print(f"Number of expressed genes in receiver: {len(expressed_genes_receiver)}")

In [ ]:
# Get expressed receptors and define potential ligands (sender-agnostic)
all_receptors = lr_network["to"].unique().tolist()
expressed_receptors = list(set(all_receptors) & set(expressed_genes_receiver))

potential_ligands = (
    lr_network[lr_network["to"].isin(expressed_receptors)]["from"]
    .unique()
    .tolist()
)
print(f"Number of potential ligands (sender-agnostic): {len(potential_ligands)}")

In [ ]:
# Sender-focused: define sender cell types and their expressed genes
sender_celltypes = ["CD4 T", "Treg", "Mono", "NK", "B", "DC"]

expressed_genes_sender = set()
for ct in sender_celltypes:
    genes = nn.get_expressed_genes(adata, celltype_col="celltype", celltype=ct, pct=0.05)
    expressed_genes_sender.update(genes)
expressed_genes_sender = list(expressed_genes_sender)

# Filter potential ligands to those expressed in sender cells
potential_ligands_focused = list(set(potential_ligands) & set(expressed_genes_sender))

print(f"Expressed genes in senders: {len(expressed_genes_sender)}")
print(f"Potential ligands (agnostic): {len(potential_ligands)}")
print(f"Potential ligands (sender-focused): {len(potential_ligands_focused)}")

### 2. Define the gene set of interest

The gene set of interest are the differentially expressed (DE) genes between the two conditions in the receiver cell type. The condition of interest is 'LCMV', the reference condition is 'SS'. We use Scanpy's Wilcoxon test for differential expression.

In [ ]:
condition_oi = "LCMV"
condition_reference = "SS"

# Subset to receiver cells
receiver_mask = adata.obs["celltype"] == receiver
adata_receiver = adata[receiver_mask].copy()

# Run DE analysis
sc.tl.rank_genes_groups(
    adata_receiver,
    groupby="aggregate",
    groups=[condition_oi],
    reference=condition_reference,
    method="wilcoxon",
)

de_table = sc.get.rank_genes_groups_df(adata_receiver, group=condition_oi)
de_table = de_table.rename(columns={"names": "gene", "logfoldchanges": "avg_log2FC", "pvals_adj": "p_val_adj"})

# Filter DE genes
geneset_oi = de_table[
    (de_table["p_val_adj"] <= 0.05) & (de_table["avg_log2FC"].abs() >= 0.25)
]["gene"].tolist()

# Keep only genes in the ligand-target matrix
geneset_oi = [g for g in geneset_oi if g in ligand_target_matrix.rownames]
print(f"Number of DE genes in gene set: {len(geneset_oi)}")

### 3. Define the background genes

All expressed genes in the receiver cell population (that are also in the ligand-target matrix) form the background set.

In [ ]:
background_expressed_genes = [
    g for g in expressed_genes_receiver if g in ligand_target_matrix.rownames
]

print(f"Background expressed genes: {len(background_expressed_genes)}")
print(f"Gene set of interest: {len(geneset_oi)}")

### 4. Perform NicheNet ligand activity analysis

This is the main step of NicheNet where the potential ligands are ranked based on the presence of their target genes in the gene set of interest. Ligands are ranked by the area under the precision-recall curve (AUPR).

We first show the results of the **sender-agnostic** approach.

In [ ]:
ligand_activities = nn.predict_ligand_activities(
    geneset=geneset_oi,
    background_expressed_genes=background_expressed_genes,
    ligand_target_matrix=ligand_target_matrix,
    potential_ligands=potential_ligands,
)

ligand_activities = ligand_activities.sort_values("aupr_corrected", ascending=False)
ligand_activities["rank"] = ligand_activities["aupr_corrected"].rank(ascending=False)
ligand_activities.head(30)

In [ ]:
# Select top 30 ligands
best_upstream_ligands = (
    ligand_activities.nlargest(30, "aupr_corrected")["test_ligand"].tolist()
)

In [ ]:
# Ligand activity histogram
cutoff_30 = ligand_activities.nlargest(30, "aupr_corrected")["aupr_corrected"].min()

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(ligand_activities["aupr_corrected"], bins=30, color="darkorange", edgecolor="black")
ax.axvline(cutoff_30, color="red", linestyle="--", linewidth=1)
ax.set_xlabel("Ligand activity (AUPR corrected)")
ax.set_ylabel("# ligands")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize ligand activity of top 30 ligands as a heatmap
vis_ligand_aupr = (
    ligand_activities[ligand_activities["test_ligand"].isin(best_upstream_ligands)]
    .set_index("test_ligand")[["aupr_corrected"]]
    .sort_values("aupr_corrected")
)

nn.make_heatmap_ggplot(
    vis_ligand_aupr,
    y_name="Prioritized ligands",
    x_name="Ligand activity",
    legend_title="AUPR",
    color="darkorange",
    x_axis=False,
    figsize=(3, 8),
    show=True,
)

### 5. Infer target genes and receptors of top-ranked ligands

#### Active target gene inference

Active target genes are genes in the gene set of interest that have the highest regulatory potential for each top-ranked ligand.

In [ ]:
# Get weighted ligand-target links for all top ligands
active_ligand_target_links_df = pd.concat(
    [
        nn.get_weighted_ligand_target_links(
            ligand_oi=lig,
            geneset=geneset_oi,
            ligand_target_matrix=ligand_target_matrix,
            n=100,
        )
        for lig in best_upstream_ligands
    ],
    ignore_index=True,
).dropna()

print(f"Number of ligand-target links: {len(active_ligand_target_links_df)}")
active_ligand_target_links_df.head()

In [ ]:
# Prepare visualization matrix with cutoff
active_ligand_target_links = nn.prepare_ligand_target_visualization(
    ligand_target_df=active_ligand_target_links_df,
    ligand_target_matrix=ligand_target_matrix,
    cutoff=0.33,
)

# The returned array has .rownames (targets) and .colnames (ligands)
order_ligands = [l for l in reversed(best_upstream_ligands) if l in active_ligand_target_links.colnames]
order_targets = [
    t for t in active_ligand_target_links_df["target"].unique()
    if t in active_ligand_target_links.rownames
]

# Build visualization DataFrame (ligands as rows, targets as columns)
df_full = pd.DataFrame(
    active_ligand_target_links,
    index=active_ligand_target_links.rownames,
    columns=active_ligand_target_links.colnames,
)
vis_ligand_target = df_full.loc[order_targets, order_ligands].T

nn.make_heatmap_ggplot(
    vis_ligand_target,
    y_name="Prioritized ligands",
    x_name="Predicted target genes",
    color="purple",
    legend_title="Regulatory potential",
    figsize=(12, 8),
    show=True,
)

#### Receptors of top-ranked ligands

We identify which receptors have the highest interaction potential with the top-ranked ligands.

In [ ]:
ligand_receptor_links_df = nn.get_weighted_ligand_receptor_links(
    best_upstream_ligands, expressed_receptors, lr_network, weighted_networks["lr_sig"]
)

vis_ligand_receptor_network = nn.prepare_ligand_receptor_visualization(
    ligand_receptor_links_df, best_upstream_ligands, order_hclust="both"
)

# Transpose: ligands as rows, receptors as columns
df_lr = pd.DataFrame(
    vis_ligand_receptor_network,
    index=vis_ligand_receptor_network.rownames,
    columns=vis_ligand_receptor_network.colnames,
).T

nn.make_heatmap_ggplot(
    df_lr,
    y_name="Ligands",
    x_name="Receptors",
    color="mediumvioletred",
    legend_title="Prior interaction potential",
    figsize=(10, 8),
    show=True,
)

### 6. Sender-focused approach

To perform the sender-focused approach, we subset the ligand activities to only contain expressed ligands from the defined sender populations.

In [ ]:
# Save sender-agnostic results
ligand_activities_all = ligand_activities.copy()
best_upstream_ligands_all = best_upstream_ligands.copy()

# Filter to sender-expressed ligands
ligand_activities = ligand_activities[
    ligand_activities["test_ligand"].isin(potential_ligands_focused)
]

best_upstream_ligands = (
    ligand_activities.nlargest(30, "aupr_corrected")["test_ligand"].unique().tolist()
)

In [ ]:
# Sender-focused ligand activity heatmap
vis_ligand_aupr = (
    ligand_activities[ligand_activities["test_ligand"].isin(best_upstream_ligands)]
    .set_index("test_ligand")[["aupr_corrected"]]
    .sort_values("aupr_corrected")
)

nn.make_heatmap_ggplot(
    vis_ligand_aupr,
    y_name="Prioritized ligands",
    x_name="Ligand activity",
    legend_title="AUPR",
    color="darkorange",
    x_axis=False,
    figsize=(3, 8),
    show=True,
)

In [ ]:
# Sender-focused: ligand-target heatmap
active_ligand_target_links_df = pd.concat(
    [
        nn.get_weighted_ligand_target_links(
            ligand_oi=lig,
            geneset=geneset_oi,
            ligand_target_matrix=ligand_target_matrix,
            n=100,
        )
        for lig in best_upstream_ligands
    ],
    ignore_index=True,
).dropna()

active_ligand_target_links = nn.prepare_ligand_target_visualization(
    ligand_target_df=active_ligand_target_links_df,
    ligand_target_matrix=ligand_target_matrix,
    cutoff=0.33,
)

order_ligands = [l for l in reversed(best_upstream_ligands) if l in active_ligand_target_links.colnames]
order_targets = [
    t for t in active_ligand_target_links_df["target"].unique()
    if t in active_ligand_target_links.rownames
]

df_full = pd.DataFrame(
    active_ligand_target_links,
    index=active_ligand_target_links.rownames,
    columns=active_ligand_target_links.colnames,
)
vis_ligand_target = df_full.loc[order_targets, order_ligands].T

nn.make_heatmap_ggplot(
    vis_ligand_target,
    y_name="Prioritized ligands",
    x_name="Predicted target genes",
    color="purple",
    legend_title="Regulatory potential",
    figsize=(12, 8),
    show=True,
)

In [ ]:
# Sender-focused: ligand-receptor heatmap
ligand_receptor_links_df = nn.get_weighted_ligand_receptor_links(
    best_upstream_ligands, expressed_receptors, lr_network, weighted_networks["lr_sig"]
)

vis_ligand_receptor_network = nn.prepare_ligand_receptor_visualization(
    ligand_receptor_links_df, best_upstream_ligands, order_hclust="both"
)

df_lr = pd.DataFrame(
    vis_ligand_receptor_network,
    index=vis_ligand_receptor_network.rownames,
    columns=vis_ligand_receptor_network.colnames,
).T

nn.make_heatmap_ggplot(
    df_lr,
    y_name="Ligands",
    x_name="Receptors",
    color="mediumvioletred",
    legend_title="Prior interaction potential",
    figsize=(10, 8),
    show=True,
)

### Visualizing expression and log-fold change in sender cells

For the sender-focused approach, we can investigate which sender cell populations are potentially the true senders of these ligands. First, we check expression via a dot plot, then compute log-fold changes between conditions per sender cell type.

In [ ]:
# Dotplot of top ligands in sender cell types
sender_mask = adata.obs["celltype"].isin(sender_celltypes)
sc.pl.dotplot(
    adata[sender_mask],
    var_names=list(reversed(best_upstream_ligands)),
    groupby="celltype",
)

In [ ]:
# Compute log-fold change of ligands between conditions per sender cell type
vis_ligand_lfc = nn.get_lfc_celltype(
    adata,
    celltype_col="celltype",
    senders=sender_celltypes,
    condition_col="aggregate",
    condition_oi=condition_oi,
    condition_ref=condition_reference,
    ligands_oi=best_upstream_ligands,
)

# Prepare as matrix for heatmap
lfc_matrix = vis_ligand_lfc.set_index("gene")
lfc_matrix = lfc_matrix.reindex(list(reversed(best_upstream_ligands)))

nn.make_threecolor_heatmap_ggplot(
    lfc_matrix,
    y_name="Prioritized ligands",
    x_name="LFC in Sender",
    low_color="midnightblue",
    mid_color="white",
    mid=float(np.nanmedian(lfc_matrix.values)),
    high_color="red",
    legend_title="LFC",
    figsize=(8, 8),
    show=True,
)

In [ ]:
# Compare sender-agnostic and sender-focused rankings
nn.make_line_plot(
    ligand_activities=ligand_activities_all,
    potential_ligands=potential_ligands_focused,
    show=True,
)

## Other follow-up analyses

- **Signaling paths**: Infer possible signaling paths between ligands and targets of interest (see `ligand_target_signaling_path` tutorial).
- **Target prediction evaluation**: Assess how well top-ranked ligands can predict the gene set of interest (see `target_prediction_evaluation_geneset` tutorial).
- **Circos plots**: Visualize ligand-target links between multiple interacting cells (see `circos` and `seurat_wrapper_circos` tutorials).